Objective:
    Observe the execution plan in Spark

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
%run ./Project_DailyFranchiseSales/00_functions

In [0]:
# import silver tables 

transactions_df = spark.read.table('workspace.bakehouse_training.silver_transactions_fact_inc')
franchises_df = spark.read.table('workspace.bakehouse_training.silver_franchises_dim')

In [0]:
# This function is used to enrich silver transactions tables with franchises
enriched_df = enrich_with_franchises(
    transactions_df.drop('customer_exists', 'franchise_exists', 'dq_reason', 'dq_status', 'checked_at', '_batch_id', '_run_id'),
    franchises_df,
    cols_franchises = ['franchiseID', 'franchiseName', 'city', 'district', 'country', 'size']
)

In [0]:
# Inspect the plan Databricks + Photon 
enriched_df.explain('formatted')

Databricks choose automatically Boradcast Hash Join in : PhotonBroadcastHashJoin LeftOuter, franchises_dim table is small enough to be distributed in executors for joining transactions table.

In [0]:
enriched_df.unpersist()

In [0]:
daily_sales_franchises_metrics = (
    enriched_df
    .groupBy('franchiseID', 'franchiseName', 'transaction_date')
    .agg(F.sum('totalPrice').alias('sum_revenue'),
        F.round(F.avg('totalPrice'), 2).alias('avg_revenue'),
        F.countDistinct('customerID').alias('amount_of_customers')
        )
    )
daily_sales_franchises_metrics = (
    daily_sales_franchises_metrics.withColumn(
        'rank_sum_revenue',
        F.dense_rank().over(Window.partitionBy('transaction_date').orderBy(F.desc('sum_revenue')))
    )
)

In [0]:
daily_sales_franchises_metrics.explain('formatted')

- Spark automatically applies column pruning:
  only 4 transaction columns and 2 franchise columns are scanned.
- Photon automatically chose a BroadcastHashJoin for the small
  franchises dimension.
- 1st Exchange: broadcasts the franchises table for the join.
- 2nd Exchange: redistributes intermediate aggregation data using
  'franchiseID', 'franchiseName', 'transaction_date' and 'customerID'.
  'customerID' is included because of .countDistinct('customerID').
- 3rd Exchange: redistributes data using the final groupBy keys
  ('franchiseID', 'franchiseName', 'transaction_date') to finalize
  the aggregation.
- 4th Exchange: partitions data by 'transaction_date' for the
  window function.
- Sort: orders sum_revenue DESC within 'transaction_date' before
  dense_rank().